# EdoFarm AI: Bilingual Agricultural Advisory Prototype

**Project Category:** Bilingual Conversational Assistant  
**Project Status:** Research prototype  
**Languages:** English and Edo (Bini)

This notebook demonstrates the current EdoFarm AI pipeline. It loads the trained intent classifier, predicts the likely crop-production topic of a farmer's question, and sends the question and predicted intent to Gemini Flash for response generation.

> This is not a fine-tuned small language model. The classifier and the generative response model are separate components.

## Problem

Smallholder farmers need understandable information about crop production, but access to agricultural extension support can be limited. Language can create an additional barrier when useful information is available mainly in English.

## What You'll Build

A prototype bilingual pipeline with two main components:

1. **Intent classification:** a trained scikit-learn pipeline using TF-IDF word and character n-grams with Logistic Regression. It predicts one of the supported crop-production intents or `out_of_scope`.
2. **Response generation:** Gemini Flash receives the farmer's question, the predicted intent, the requested response language, and safety instructions, then generates a response.

```text
Farmer question + requested language
                |
                v
       TF-IDF intent classifier
                |
                v
     Predicted intent (routing hint)
                |
                v
       Gemini Flash response generation
                |
                v
       Cautious English / Edo response
```

No approved retrieval knowledge base is connected in this prototype. Gemini is not being supplied with verified extension passages by this notebook.

## Research Question

> Can a lightweight bilingual intent classifier help route crop-production questions to a generative model, and what reliability gaps remain across English and Edo (Bini)?

## What You'll Learn

- Load and inspect a previously exported scikit-learn intent classifier.
- Run intent predictions on English and Edo (Bini) questions.
- Interpret prediction scores without treating them as proof of correctness.
- Connect the classifier to Gemini Flash for answer generation.
- Review held-out classifier results and known failure cases.
- Identify the additional human and agricultural validation required before deployment.

## Setup

Run this notebook from the **root of the EdoFarm AI project**, or set `PROJECT_ROOT` below to the folder that contains the exported classifier and project files.

Before running:

- Keep the trained classifier artifact produced by the model-training notebook in the project's `models/` directory (or set its exact path below).
- Install the same compatible scikit-learn version used when the artifact was exported. Loading a joblib/pickle model across incompatible package versions may fail.
- Create a Gemini API key in [Google AI Studio](https://aistudio.google.com/api-keys).
- Store the key in your environment as `GEMINI_API_KEY`. In Google Colab, add it under **Secrets** and enable notebook access. Never paste the key into a code cell or commit it to GitHub.
- Set `GEMINI_MODEL` to the Gemini Flash model ID that you have verified with your API key. The default below follows the current Google Gen AI SDK documentation; use your already-working model ID if it differs.

A GPU is **not required** for this notebook. The classifier runs on CPU; response generation uses the Gemini API.

Official SDK reference: [Google Gen AI Python SDK](https://googleapis.github.io/python-genai/).

## Dataset and Model Notes

The bilingual classifier dataset was derived from two public agricultural Q&A sources and prepared for intent classification.

- 306 underlying question records were expanded into 612 language examples: 306 English and 306 machine-generated Edo (Bini) examples.
- The underlying question groups were kept within a single split to reduce paraphrase leakage.
- Classifier split sizes: 392 training examples, 98 validation examples, and 122 test examples.
- The Edo translations were generated with Gemini and have **not** yet been validated by a native Bini speaker.
- The records and model outputs have **not** been validated by a qualified agricultural expert.

The seven crop-production intents are `crop_cultivation`, `pest_management`, `disease_management`, `soil_management`, `irrigation`, `fertilizer_application`, and `harvesting`. `out_of_scope` is used for questions outside the project's crop-production scope, including veterinary and livestock-health questions.

See `DATASET_CARD.md` and `MODEL_CARD.md` for the fuller dataset and model documentation.

## Important Safety and Use Note

This is a research prototype, not a production farming-advice service.

- A predicted intent is only a model output. It can be wrong, and its confidence score is not necessarily a calibrated probability of correctness.
- The classifier has demonstrated weaknesses, including missed `crop_cultivation` examples.
- Gemini may generate plausible but incorrect agricultural information. This notebook does not connect an approved, source-verified RAG knowledge base.
- Do not rely on the prototype for definitive disease diagnosis, pesticide selection, chemical dosage, or fertilizer application rates.
- High-risk or uncertain cases should be checked with a qualified agricultural extension officer.
- Edo (Bini) output must be reviewed by a competent Bini speaker before it is treated as reliable language support.
- Do not use personal farmer information or secret credentials in notebook demonstrations.

---

## Install Required Libraries

In [ ]:
%pip install -q --upgrade pandas numpy scikit-learn joblib matplotlib google-genai

## Import Libraries

In [ ]:
import os
import getpass
import warnings
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
)

from google import genai
from google.genai import types

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 180)

print("Imports completed.")

## Settings

In [ ]:
# Run the notebook from the EdoFarm AI project root.
PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_FILE = PROCESSED_DIR / "classifier_bilingual_train_provisional_v1.csv"
VALIDATION_FILE = PROCESSED_DIR / "classifier_bilingual_validation_provisional_v1.csv"
TEST_FILE = PROCESSED_DIR / "classifier_bilingual_test_provisional_v1.csv"

MODEL_OUTPUT_DIR = PROJECT_ROOT / "models"
MODEL_OUTPUT_PATH = MODEL_OUTPUT_DIR / "edofarm_intent_classifier_notebook_v1.joblib"

# Use the same Gemini Flash model ID that you have successfully tested.
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
MAX_OUTPUT_TOKENS = 700
TEMPERATURE = 0.2
RANDOM_SEED = 42

print("Project root:", PROJECT_ROOT.resolve())
print("Processed data directory:", PROCESSED_DIR.resolve())
print("Gemini model:", GEMINI_MODEL)

## Load the Bilingual Classifier Splits

The notebook uses the group-aware CSV splits created during EdoFarm AI data preparation. It does not rebuild the original source dataset or translate records again.

Expected files:

- `classifier_bilingual_train_provisional_v1.csv`
- `classifier_bilingual_validation_provisional_v1.csv`
- `classifier_bilingual_test_provisional_v1.csv`

The English and machine-generated Edo (Bini) examples from the same underlying record must remain in the same split.

In [ ]:
required_files = [TRAIN_FILE, VALIDATION_FILE, TEST_FILE]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "The following processed split files were not found:\n- "
        + "\n- ".join(missing_files)
        + "\nRun this notebook from the EdoFarm AI project root or update PROCESSED_DIR."
    )

train_df = pd.read_csv(TRAIN_FILE)
validation_df = pd.read_csv(VALIDATION_FILE)
test_df = pd.read_csv(TEST_FILE)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)
print("\nTrain columns:", train_df.columns.tolist())
train_df.head(3)

## Check and Standardise the Split Columns

The preprocessing notebook may retain additional provenance fields. This cell locates the text, intent, language, and split-group columns without discarding the original dataframes.

In [ ]:
def find_column(frame: pd.DataFrame, candidates, description: str, required=True):
    """Return the first matching column name from a list of common alternatives."""
    by_lower = {str(col).strip().lower(): col for col in frame.columns}
    for candidate in candidates:
        if candidate.lower() in by_lower:
            return by_lower[candidate.lower()]
    if required:
        raise KeyError(
            f"Could not find the {description} column. Available columns: {frame.columns.tolist()}"
        )
    return None

TEXT_CANDIDATES = ["question_text", "question", "text", "input_text", "utterance", "query", "message"]
LABEL_CANDIDATES = ["primary_intent", "intent", "label", "target"]
LANGUAGE_CANDIDATES = ["language", "language_name", "language_code"]
GROUP_CANDIDATES = ["split_group_id", "question_group", "duplicate_group_id", "group_id"]

TEXT_COL = find_column(train_df, TEXT_CANDIDATES, "question/text")
LABEL_COL = find_column(train_df, LABEL_CANDIDATES, "intent/label")
LANGUAGE_COL = find_column(train_df, LANGUAGE_CANDIDATES, "language", required=False)
GROUP_COL = find_column(train_df, GROUP_CANDIDATES, "split-group", required=False)

for split_name, frame in (("validation", validation_df), ("test", test_df)):
    if TEXT_COL not in frame.columns or LABEL_COL not in frame.columns:
        raise KeyError(
            f"The {split_name} split does not contain {TEXT_COL!r} and {LABEL_COL!r}. "
            f"Columns found: {frame.columns.tolist()}"
        )

# Keep only rows with non-empty input text and labels; report any rows removed.
def clean_split(frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    cleaned = frame.copy()
    before = len(cleaned)
    cleaned[TEXT_COL] = cleaned[TEXT_COL].fillna("").astype(str).str.strip()
    cleaned[LABEL_COL] = cleaned[LABEL_COL].fillna("").astype(str).str.strip()
    cleaned = cleaned[(cleaned[TEXT_COL] != "") & (cleaned[LABEL_COL] != "")].copy()
    print(f"{split_name}: retained {len(cleaned)}/{before} rows")
    return cleaned

train_df = clean_split(train_df, "train")
validation_df = clean_split(validation_df, "validation")
test_df = clean_split(test_df, "test")

print("\nDetected text column:", TEXT_COL)
print("Detected label column:", LABEL_COL)
print("Detected language column:", LANGUAGE_COL)
print("Detected group column:", GROUP_COL)

In [ ]:
# Check for split-group overlap when the provenance field is available.
if GROUP_COL:
    train_groups = set(train_df[GROUP_COL].dropna().astype(str))
    validation_groups = set(validation_df[GROUP_COL].dropna().astype(str))
    test_groups = set(test_df[GROUP_COL].dropna().astype(str))
    overlaps = {
        "train_validation": train_groups & validation_groups,
        "train_test": train_groups & test_groups,
        "validation_test": validation_groups & test_groups,
    }
    for pair, overlap in overlaps.items():
        print(f"{pair} group overlap: {len(overlap)}")
    if any(overlaps.values()):
        raise ValueError("Split-group leakage detected. Recreate the group-aware splits before training.")
else:
    print("No split-group column found; this notebook cannot independently verify group leakage.")

print("\nIntent distribution in training split:")
display(train_df[LABEL_COL].value_counts().rename("examples").to_frame())

if LANGUAGE_COL:
    print("\nTraining examples by language:")
    display(pd.crosstab(train_df[LANGUAGE_COL], train_df[LABEL_COL]))

## Intent Definitions and Scope

| Intent | Intended meaning |
|---|---|
| `crop_cultivation` | Planting, spacing, crop establishment and general crop-growing practices |
| `pest_management` | Crop pest prevention, monitoring and management |
| `disease_management` | Crop disease symptoms, prevention and management |
| `soil_management` | Soil preparation, soil health and soil conservation |
| `irrigation` | Crop water needs, watering and drainage |
| `fertilizer_application` | Nutrient management and fertilizer use |
| `harvesting` | Harvest timing, maturity and post-harvest handling |
| `out_of_scope` | Requests outside the project's crop-production scope, such as veterinary and livestock-health questions |

These are classification labels, not a guarantee that a question or an answer is agriculturally correct.

## Build Candidate TF-IDF Classifiers

The selected approach in the earlier experiment combined word and character n-gram features. Here, three transparent baselines are compared on the **validation split**:

- word n-grams;
- character n-grams;
- combined word and character n-grams.

The Logistic Regression classifier is fitted only on the training split during model selection. The test split is kept separate until the feature configuration has been selected.

In [ ]:
def make_classifier(feature_mode: str) -> Pipeline:
    """Build a TF-IDF + Logistic Regression intent classifier."""
    word_features = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
    )
    character_features = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=1,
        sublinear_tf=True,
    )

    if feature_mode == "word":
        features = word_features
    elif feature_mode == "character":
        features = character_features
    elif feature_mode == "word_character_combined":
        features = FeatureUnion([
            ("word", word_features),
            ("character", character_features),
        ])
    else:
        raise ValueError(f"Unknown feature mode: {feature_mode}")

    return Pipeline([
        ("features", features),
        ("classifier", LogisticRegression(max_iter=3000, random_state=RANDOM_SEED)),
    ])

X_train = train_df[TEXT_COL].tolist()
y_train = train_df[LABEL_COL].tolist()
X_validation = validation_df[TEXT_COL].tolist()
y_validation = validation_df[LABEL_COL].tolist()

print("Training and validation examples:", len(X_train), len(X_validation))

## Select the Feature Configuration on Validation Data

Macro F1 gives each intent equal weight, so it helps reveal when a model performs poorly on less frequent classes. The validation result is used to select the feature configuration; do not select the model by looking at test performance.

In [ ]:
validation_rows = []
validation_models = {}

for feature_mode in ("word", "character", "word_character_combined"):
    candidate = make_classifier(feature_mode)
    candidate.fit(X_train, y_train)
    val_predictions = candidate.predict(X_validation)
    val_macro_f1 = f1_score(y_validation, val_predictions, average="macro", zero_division=0)
    val_accuracy = accuracy_score(y_validation, val_predictions)
    validation_models[feature_mode] = candidate
    validation_rows.append({
        "feature_mode": feature_mode,
        "validation_accuracy": val_accuracy,
        "validation_macro_f1": val_macro_f1,
    })

validation_results = pd.DataFrame(validation_rows).sort_values(
    "validation_macro_f1", ascending=False
).reset_index(drop=True)
# Format a display-only copy as percentages; this avoids the optional jinja2 dependency.
validation_display = validation_results.copy()
validation_display["validation_accuracy"] = validation_display["validation_accuracy"].map("{:.1%}".format)
validation_display["validation_macro_f1"] = validation_display["validation_macro_f1"].map("{:.1%}".format)
display(validation_display)

BEST_FEATURE_MODE = validation_results.loc[0, "feature_mode"]
print("Selected feature configuration:", BEST_FEATURE_MODE)

## Fit the Selected Classifier and Evaluate the Held-Out Test Split

After choosing the feature configuration on validation data, refit that configuration using the combined training and validation examples. The test split is used once for the final evaluation in this notebook.

In [ ]:
train_validation_df = pd.concat([train_df, validation_df], ignore_index=True)
X_train_validation = train_validation_df[TEXT_COL].tolist()
y_train_validation = train_validation_df[LABEL_COL].tolist()
X_test = test_df[TEXT_COL].tolist()
y_test = test_df[LABEL_COL].tolist()

classifier = make_classifier(BEST_FEATURE_MODE)
classifier.fit(X_train_validation, y_train_validation)
test_predictions = classifier.predict(X_test)

print("Selected feature mode:", BEST_FEATURE_MODE)
print("Test examples:", len(test_df))
print("\nClassification report:")
print(classification_report(y_test, test_predictions, zero_division=0))

print("\nConfusion matrix (rows = true labels, columns = predicted labels):")
labels = sorted(set(y_test) | set(test_predictions))
cm = confusion_matrix(y_test, test_predictions, labels=labels)
display(pd.DataFrame(cm, index=labels, columns=labels))

In [ ]:
# Save the notebook-trained candidate and its key settings for later use.
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model_bundle = {
    "model": classifier,
    "feature_mode": BEST_FEATURE_MODE,
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "random_seed": RANDOM_SEED,
    "notes": "Retrained in notebook.ipynb; see MODEL_CARD.md before use.",
}
joblib.dump(model_bundle, MODEL_OUTPUT_PATH)
print("Saved classifier bundle:", MODEL_OUTPUT_PATH.resolve())

## Create a Prediction Helper

The function below uses the fitted classifier to predict a topic. When the estimator exposes `predict_proba`, it also displays the top class scores. These scores are diagnostics and are not necessarily calibrated probabilities of correctness.

In [ ]:
def get_classifier_classes(estimator):
    """Read class names from a fitted sklearn estimator or Pipeline."""
    classes = getattr(estimator, "classes_", None)
    if classes is not None:
        return [str(value) for value in classes]
    if hasattr(estimator, "named_steps") and estimator.named_steps:
        final_estimator = list(estimator.named_steps.values())[-1]
        classes = getattr(final_estimator, "classes_", None)
        if classes is not None:
            return [str(value) for value in classes]
    return []


CLASS_NAMES = get_classifier_classes(classifier)
print("Fitted classes:", CLASS_NAMES)


def predict_intent(question: str) -> dict:
    """Predict an intent and return the top scores when available."""
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Enter a non-empty question.")

    predicted = str(classifier.predict([question])[0])
    result = {
        "question": question,
        "predicted_intent": predicted,
        "top_predictions": [],
        "max_score": None,
        "score_note": "No predict_proba method is available on this classifier.",
    }

    if hasattr(classifier, "predict_proba"):
        probabilities = np.asarray(classifier.predict_proba([question]))[0]
        classes = get_classifier_classes(classifier)
        if len(classes) == len(probabilities):
            order = np.argsort(probabilities)[::-1]
            result["top_predictions"] = [
                {"intent": classes[i], "score": float(probabilities[i])}
                for i in order[:3]
            ]
            result["max_score"] = float(probabilities[order[0]])
            result["score_note"] = (
                "Model score only; not established as a calibrated probability of correctness."
            )
    return result


def display_prediction(result: dict):
    print("Question:", result["question"])
    print("Predicted intent:", result["predicted_intent"])
    if result["top_predictions"]:
        print("Top class scores:")
        for item in result["top_predictions"]:
            print(f"  {item['intent']:<24} {item['score']:.3f}")
    print("Note:", result["score_note"])

print("Prediction helper is ready.")

## Run Classifier Smoke Tests

In [ ]:
# These examples are demonstrations, not a replacement for evaluation on a held-out dataset.
smoke_test_questions = [
    "I want to plant maize. What should I consider before planting?",
    "How can I reduce weeds around my cassava plants?",
    "My chicken is coughing and seems weak. What should I do?",
]

smoke_test_results = []
for question in smoke_test_questions:
    result = predict_intent(question)
    smoke_test_results.append(result)
    print("-" * 72)
    display_prediction(result)

smoke_test_table = pd.DataFrame([
    {
        "question": r["question"],
        "predicted_intent": r["predicted_intent"],
        "max_score": r["max_score"],
    }
    for r in smoke_test_results
])
smoke_test_table

## Connect to Gemini Flash

The classifier predicts the topic; it does not write the advisory response. Gemini Flash is the separate generation component.

The API key is read from `GEMINI_API_KEY`. If it is not already set in the environment, the cell asks for it privately. The key is not displayed or written into the notebook.

In [ ]:
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    api_key = getpass.getpass("Enter your Gemini API key (input hidden): ")

if not api_key.strip():
    raise ValueError("A Gemini API key is required to call the response-generation model.")

client = genai.Client(api_key=api_key)
print("Gemini client initialized. The API key was not printed.")

## Define the Response-Generation Prompt

The intent prediction is included as a **routing hint**, not as unquestionable truth. This matters because a crop-production question can be misclassified as `out_of_scope`.

The prompt also sets limits: no definitive diagnosis, no invented product names, no unsupported chemical or fertilizer rates, no fabricated source citations, and referral to an extension officer when advice is uncertain or high-risk.

In [ ]:
SYSTEM_INSTRUCTIONS = """You are the response-generation component of EdoFarm AI, a research prototype for crop-production questions.

Scope:
- The project covers crop cultivation, crop pests, crop diseases, soil management, irrigation, fertilizer application, and harvesting.
- Veterinary care, livestock disease, poultry care, aquaculture, and unrelated topics are outside the project's scope.

Use the predicted intent as a hint only. Read the actual farmer question. The classifier can be wrong; do not reject an obviously crop-related question only because its predicted label is out_of_scope. If the question is outside crop production, explain the scope boundary and recommend an appropriate qualified professional where relevant.

Safety and accuracy:
- Give cautious, practical, clearly worded information.
- Do not claim to diagnose a crop disease with certainty from a short text description.
- Do not invent pesticide/fungicide/herbicide product names, dosages, mixing instructions, application schedules, or fertilizer rates.
- When a recommendation depends on crop variety, location, soil, season, or severity, say what information is missing rather than guessing.
- Do not pretend to have consulted a verified knowledge base or cite sources that were not provided.
- For uncertain or high-risk agricultural questions, advise the farmer to contact a qualified agricultural extension officer.

Language:
- Respond in the language requested by the user: English or Edo (Bini).
- Edo (Bini) generation is experimental and machine-generated. Do not claim that the wording has been checked by a native speaker.
- Keep essential technical terms in English when you cannot confidently express them in Edo (Bini), and explain them simply.
- Use clear, respectful language suitable for a smallholder farmer.

Remember: this is a research prototype, not an expert-validated agricultural advisory service."""


def _gemini_http_status(error: Exception):
    """Best-effort extraction of an HTTP status code from a Google Gen AI SDK error."""
    for attribute in ("status_code", "code"):
        value = getattr(error, attribute, None)
        if isinstance(value, int):
            return value
    response = getattr(error, "response", None)
    if response is not None:
        value = getattr(response, "status_code", None)
        if isinstance(value, int):
            return value
    return None


def _is_transient_gemini_error(error: Exception) -> bool:
    """Identify temporary overload/rate-limit/server responses that are safe to retry."""
    status = _gemini_http_status(error)
    if status in {408, 429, 500, 502, 503, 504}:
        return True
    message = f"{type(error).__name__}: {error}".upper()
    return any(marker in message for marker in (
        "UNAVAILABLE", "RESOURCE_EXHAUSTED", "TOO MANY REQUESTS",
        "SERVICE UNAVAILABLE", "TIMED OUT", "TIMEOUT",
    ))


def generate_advisory_response(
    question: str,
    language: str = "English",
    max_attempts: int = 4,
    initial_retry_delay: float = 2.0,
) -> dict:
    """Classify a question and ask Gemini Flash for an answer.

    Temporary 408/429/5xx errors are retried with exponential backoff.
    If Gemini remains unavailable, return the classifier prediction and an
    error message instead of terminating the notebook.
    """
    prediction = predict_intent(question)
    allowed_languages = {"English", "Edo (Bini)"}
    if language not in allowed_languages:
        raise ValueError(f"language must be one of: {sorted(allowed_languages)}")
    if max_attempts < 1:
        raise ValueError("max_attempts must be at least 1")

    top_scores = prediction["top_predictions"]
    score_text = (
        ", ".join(f"{x['intent']}={x['score']:.3f}" for x in top_scores)
        if top_scores else "not available"
    )

    prompt = f"""{SYSTEM_INSTRUCTIONS}

Farmer question:
{question}

Classifier's predicted intent: {prediction['predicted_intent']}
Classifier's top scores (diagnostic only): {score_text}
Requested response language: {language}

Write a helpful response. If important context is missing, ask a small number of specific follow-up questions. Do not invent exact agricultural rates or claim expert verification."""

    for attempt in range(1, max_attempts + 1):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                ),
            )
            answer = (response.text or "").strip()
            if not answer:
                return {
                    "question": question,
                    "language": language,
                    "predicted_intent": prediction["predicted_intent"],
                    "top_predictions": top_scores,
                    "answer": None,
                    "error": "Gemini returned an empty response. Check the model response and API settings.",
                }

            return {
                "question": question,
                "language": language,
                "predicted_intent": prediction["predicted_intent"],
                "top_predictions": top_scores,
                "answer": answer,
                "error": None,
            }

        except Exception as exc:
            status = _gemini_http_status(exc)
            transient = _is_transient_gemini_error(exc)
            if not transient:
                return {
                    "question": question,
                    "language": language,
                    "predicted_intent": prediction["predicted_intent"],
                    "top_predictions": top_scores,
                    "answer": None,
                    "error": (
                        f"Gemini request failed ({type(exc).__name__}): {exc}. "
                        "Check the API key, model ID, API access, and request settings."
                    ),
                }

            if attempt == max_attempts:
                return {
                    "question": question,
                    "language": language,
                    "predicted_intent": prediction["predicted_intent"],
                    "top_predictions": top_scores,
                    "answer": None,
                    "error": (
                        f"Gemini remained temporarily unavailable after {max_attempts} attempts"
                        + (f" (HTTP {status})" if status else "")
                        + f": {exc}. The intent classifier still ran successfully; retry the response-generation cell later."
                    ),
                }

            delay = initial_retry_delay * (2 ** (attempt - 1))
            print(
                f"Gemini temporarily unavailable"
                + (f" (HTTP {status})" if status else "")
                + f". Retrying in {delay:.0f} seconds "
                + f"(attempt {attempt + 1} of {max_attempts})..."
            )
            time.sleep(delay)


def display_advisory_result(result: dict) -> None:
    """Print either the generated answer or a readable API error."""
    print("Predicted intent:", result["predicted_intent"])
    if result.get("error"):
        print("\nResponse generation was not completed:")
        print(result["error"])
        return
    print("\nGenerated response:\n")
    print(result["answer"])

print("Response-generation function is ready (with transient-error retries).")


## Try an English Question

In [ ]:
english_demo = generate_advisory_response(
    "When should I apply fertilizer to maize during its growing season?",
    language="English",
)
display_advisory_result(english_demo)

## Try an Edo (Bini) Response

This example keeps the question in English and requests an Edo (Bini) response. It is a generation demonstration only. It does **not** establish that the translation is natural, complete, or suitable for farmer use. Have a competent Bini speaker review every Edo response.

In [ ]:
edo_demo = generate_advisory_response(
    "What can I do to improve my soil before planting maize?",
    language="Edo (Bini)",
)
if edo_demo.get("error"):
    display_advisory_result(edo_demo)
else:
    print("Predicted intent:", edo_demo["predicted_intent"])
    print("\nMachine-generated Edo (Bini) response — requires speaker review:\n")
    print(edo_demo["answer"])

## Optional: Ask Your Own Question

Change the text and output language in this cell. Each run calls the Gemini API and may incur usage under your API account.

In [ ]:
# Edit these two values before running.
MY_QUESTION = "My cassava leaves are becoming yellow. What should I check?"
MY_RESPONSE_LANGUAGE = "English"  # Choose: "English" or "Edo (Bini)"

my_result = generate_advisory_response(MY_QUESTION, language=MY_RESPONSE_LANGUAGE)
display_advisory_result(my_result)

## Held-Out Classifier Evaluation

The following table is calculated from the fresh notebook run's test predictions. These scores may differ from the values documented for the earlier selected classifier in `MODEL_CARD.md`, because this notebook explicitly records and retrains its own baseline configuration.

In [ ]:
def calculate_metrics(y_true, y_pred):
    return {
        "Test examples": len(y_true),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

metric_rows = []
if LANGUAGE_COL:
    for language in sorted(test_df[LANGUAGE_COL].dropna().astype(str).unique()):
        mask = test_df[LANGUAGE_COL].astype(str).eq(language).to_numpy()
        row = {"Language": language}
        row.update(calculate_metrics(np.array(y_test)[mask], np.array(test_predictions)[mask]))
        metric_rows.append(row)
else:
    row = {"Language": "All languages"}
    row.update(calculate_metrics(y_test, test_predictions))
    metric_rows.append(row)

test_metrics = pd.DataFrame(metric_rows).set_index("Language")
# Format a display-only copy as percentages; this avoids the optional jinja2 dependency.
test_metrics_display = test_metrics.apply(lambda column: column.map("{:.1%}".format))
display(test_metrics_display)

### Compare English and Edo (Bini) Test Performance

In [ ]:
metric_columns = ["Accuracy", "Macro precision", "Macro recall", "Macro F1", "Weighted F1"]
ax = test_metrics[metric_columns].T.plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1),
    title="Selected bilingual intent classifier — held-out test results",
    rot=20,
)
ax.set_ylabel("Score")
ax.set_xlabel("Metric")
plt.tight_layout()
plt.show()

### What the Metrics Do Not Show

- The held-out test split is small. A single prediction can materially change the reported percentages.
- Inspect per-class precision, recall, and support in the classification report above; an acceptable overall accuracy can hide a class that the model fails to detect.
- Edo (Bini) examples in this dataset are machine-generated translations, not naturally collected farmer messages, and have not yet been reviewed by a native speaker.
- The metrics measure intent prediction, not the factual quality, usefulness, or safety of Gemini's generated agricultural responses.
- The response-generation component needs a separate evaluation involving agricultural experts and competent Bini speakers.
- Compare the results from this fresh baseline with the previously selected model's results in `MODEL_CARD.md`; do not assume that a new run exactly reproduces the earlier artifact.

## Review Outputs and Prepare for Human Language Review

For a small manual review exercise, save model-generated responses and have reviewers record corrections. A Bini speaker should review Edo wording; an agricultural expert should review agronomic correctness. These are separate review tasks.

In [ ]:
# Save the two demonstration outputs as a review template.
review_rows = []
for item in (english_demo, edo_demo):
    review_rows.append({
        "question": item["question"],
        "requested_language": item["language"],
        "predicted_intent": item["predicted_intent"],
        "generated_response": item["answer"],
        "bini_speaker_review": "not_reviewed",
        "agricultural_expert_review": "not_reviewed",
        "corrected_response": "",
        "review_notes": "",
    })

review_template = pd.DataFrame(review_rows)
review_template.to_csv("edofarm_response_review_template.csv", index=False)
print("Saved: edofarm_response_review_template.csv")
review_template

## Reflect

Use the results, smoke-test predictions, and generated responses to answer:

- Did the classifier assign sensible intents to the sample questions? Which mistakes could cause a farmer to receive an irrelevant response?
- Why should a predicted class and its score be treated as a hint rather than as proof?
- What does the zero recall for `crop_cultivation` mean for the current prototype?
- Does a response sound helpful while still avoiding unsupported fertilizer or pesticide rates?
- Can a generated Edo (Bini) response be trusted without review by a competent speaker? Why not?
- What evidence would be needed before this assistant could be tested with farmers?
- Which risks come from the classifier, which come from Gemini, and which come from the source dataset?

## What Next?

Possible next steps for the project:

- Increase and rebalance the intent dataset, especially `crop_cultivation`, using carefully reviewed examples.
- Conduct native Bini-speaker review of the machine-generated translations and collect natural Bini questions with appropriate consent.
- Arrange agricultural expert review of crop-production questions, reference answers, and generated responses.
- Perform class-by-class error analysis using saved test predictions, not only aggregate metrics.
- Add a retrieval layer only after extension-guide passages have been extracted, source-checked, and approved for use.
- Evaluate answer quality separately from intent accuracy, using agricultural reviewers and bilingual human preference tests.
- Integrate the classifier and Gemini call behind the JavaScript backend; keep API keys on the server, not in the browser.
- Test with a controlled pilot and a clear human escalation process before considering deployment.

## References

- [FarmerChat Nigeria dataset](https://huggingface.co/datasets/DigiGreen/farmerchat-queries)
- [Agronomy QA dataset](https://huggingface.co/datasets/flamiinngo/agronomy-qa-agriculture)
- [Google Gen AI Python SDK documentation](https://googleapis.github.io/python-genai/)
- Project documentation: `DATASET_CARD.md` and `MODEL_CARD.md`

**Disclaimer:** This notebook is for academic research and prototyping. It does not provide validated agricultural recommendations.